In [3]:
import confnotebook

In [4]:
from pathlib import Path

source = Path("../examples/RPA-6542/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 18470938
[1] 18470982
[2] 18548791
[3] 18561292
[4] 18561867
[5] 18639563
[6] 18660877
[7] 18667876
[8] 18667914
[9] 18668755
[10] 18674893
[11] 18687424
[12] 18690095
[13] 18690959
[14] 18692621
[15] 18699963
[16] 18892339
[17] 18892958
[18] 18892969
[19] 18897098
[20] 7-1
[21] [Untitled]_23-48


In [5]:
IDX_FILE = 9

In [6]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline()
file = files[IDX_FILE]
document = pipeline.build(file.read_bytes())

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon

In [7]:
from app.infrastructure.services.extractor.period_ext import (
    _collect_paragraph_text,
    _collect_table_text,
    extract_period,
)

summary_text_table = _collect_table_text(document)
summary_text_paragraph = _collect_paragraph_text(document)

print(summary_text_paragraph)

Акт сверки взаимных расчетов за период: Январь 2025 г. - Март 2026 г. между О0О "КЛАПЕРОН" (ИНН 9701222720) и ООО "РУССКИЙ РАДИАТОР" (ИНН 1006013150) Мы, нижеподписавшиеся, Генеральный Директор ООО "КЛАПЕРОН" Баринова Ольга Владимировна, с одной стороны, и Генеральный Директор ООО "РуСскИй РАдИАТОР" Колпаков Никита Сергеевич, с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее: на 31.03.2026 задолженность в пользу О00 "КЛАПЕРОН" 115 800,00 руб. (Сто пятнадцать тысяч восемьсот рублей 00 копеек). От ООО "КЛАПЕРОН" От ООО "РУССКИЙ РАДИАТОР" Генеральный Директор Генеральный Директор (Баринова О. В.) (Колпаков Н. С.) М.П. М.П.


In [8]:
from extractor.process import extract

res = extract(summary_text_paragraph)

2026-07-19 22:07:47.703 | DEBUG    | extractor.process:extract:19 - Текст до нормализации: Акт сверки взаимных расчетов за период: Январь 2025 г. - Март 2026 г. между О0О "КЛАПЕРОН" (ИНН 9701222720) и ООО "РУССКИЙ РАДИАТОР" (ИНН 1006013150) Мы, нижеподписавшиеся, Генеральный Директор ООО "КЛАПЕРОН" Баринова Ольга Владимировна, с одной стороны, и Генеральный Директор ООО "РуСскИй РАдИАТОР" Колпаков Никита Сергеевич, с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее: на 31.03.2026 задолженность в пользу О00 "КЛАПЕРОН" 115 800,00 руб. (Сто пятнадцать тысяч восемьсот рублей 00 копеек). От ООО "КЛАПЕРОН" От ООО "РУССКИЙ РАДИАТОР" Генеральный Директор Генеральный Директор (Баринова О. В.) (Колпаков Н. С.) М.П. М.П.
2026-07-19 22:07:47.704 | DEBUG    | extractor.process:extract:21 - Текст после нормализации: АКТ СВЕРКИ ВЗАИМНЫХ РАСЧЕТОВ ЗА ПЕРИОД: ЯНВАРЬ 2025 Г. - МАРТ 2026 Г. МЕЖДУ ООО "КЛАПЕРОН" (ИНН 9701222720) И ООО "РУССКИЙ 

In [9]:
for t in res.tokens:
    print(t)

DateReference(token=Token(start=40, end=68, text='ЯНВАРЬ 2025 Г. - МАРТ 2026 Г'), date='01.01.2025', date_end='31.03.2026')
OrganizationReference(token=Token(start=76, end=90, text='ООО "КЛАПЕРОН"'), name='КЛАПЕРОН', org_form='ООО')
OrganizationReference(token=Token(start=110, end=132, text='ООО "РУССКИЙ РАДИАТОР"'), name='РУССКИЙ РАДИАТОР', org_form='ООО')
OrganizationReference(token=Token(start=194, end=208, text='ООО "КЛАПЕРОН"'), name='КЛАПЕРОН', org_form='ООО')
OrganizationReference(token=Token(start=278, end=300, text='ООО "РУССКИЙ РАДИАТОР"'), name='РУССКИЙ РАДИАТОР', org_form='ООО')
DateReference(token=Token(start=446, end=456, text='31.03.2026'), date='31.03.2026', date_end=None)
OrganizationReference(token=Token(start=480, end=494, text='ООО "КЛАПЕРОН"'), name='КЛАПЕРОН', org_form='ООО')
CurrencyReference(token=Token(start=495, end=505, text='115 800,00'), value=115800.0)
DigitalReference(token=Token(start=550, end=552, text='00'), value=0.0)
OrganizationReference(token=Token

In [10]:
period = extract_period(document)

2026-07-19 22:07:47.712 | DEBUG    | extractor.process:extract:19 - Текст до нормализации: Акт сверки взаимных расчетов за период: Январь 2025 г. - Март 2026 г. между О0О "КЛАПЕРОН" (ИНН 9701222720) и ООО "РУССКИЙ РАДИАТОР" (ИНН 1006013150) Мы, нижеподписавшиеся, Генеральный Директор ООО "КЛАПЕРОН" Баринова Ольга Владимировна, с одной стороны, и Генеральный Директор ООО "РуСскИй РАдИАТОР" Колпаков Никита Сергеевич, с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее: на 31.03.2026 задолженность в пользу О00 "КЛАПЕРОН" 115 800,00 руб. (Сто пятнадцать тысяч восемьсот рублей 00 копеек). От ООО "КЛАПЕРОН" От ООО "РУССКИЙ РАДИАТОР" Генеральный Директор Генеральный Директор (Баринова О. В.) (Колпаков Н. С.) М.П. М.П.
2026-07-19 22:07:47.713 | DEBUG    | extractor.process:extract:21 - Текст после нормализации: АКТ СВЕРКИ ВЗАИМНЫХ РАСЧЕТОВ ЗА ПЕРИОД: ЯНВАРЬ 2025 Г. - МАРТ 2026 Г. МЕЖДУ ООО "КЛАПЕРОН" (ИНН 9701222720) И ООО "РУССКИЙ 

In [11]:
from app.infrastructure.services.extractor.company_ext import extract_companies

orgs = extract_companies(document)

2026-07-19 22:07:47.716 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 691 символов из 2 страниц
2026-07-19 22:07:47.717 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:93 - summary_cell_texts: ['По данным ООО "КЛАПЕРОН",руб.', 'По данным ООО "РУССКИЙ РАДИАТОР", руб.']
2026-07-19 22:07:47.717 | DEBUG    | extractor.process:extract:19 - Текст до нормализации: Акт сверки взаимных расчетов за период: Январь 2025 г. - Март 2026 г. между О0О "КЛАПЕРОН" (ИНН 9701222720) и ООО "РУССКИЙ РАДИАТОР" (ИНН 1006013150) Мы, нижеподписавшиеся, Генеральный Директор ООО "КЛАПЕРОН" Баринова Ольга Владимировна, с одной стороны, и Генеральный Директор ООО "РуСскИй РАдИАТОР" Колпаков Никита Сергеевич, с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее: на 31.03.2026 задолженность в пользу О00 "КЛАПЕРОН" 115 800,00 руб. (Сто пятнадцать тысяч восемьсот 

In [12]:
from extractor import normalize, tokenize

text = normalize.transform_text("за период: Январь 2025 г. - Март 2026 г.")
print(repr(text))
for ref in tokenize.tokenize(text):
    print(type(ref).__name__, ref.token.text, getattr(ref, "value", None) or getattr(ref, "date", None))


'ЗА ПЕРИОД: ЯНВАРЬ 2025 Г. - МАРТ 2026 Г.'
DateReference ЯНВАРЬ 2025 Г. - МАРТ 2026 Г 01.01.2025
